**Required Packages**

In [ ]:
pip install langchain langchain-community  langchain-groq  langchain-text-splitters  chromadb  faiss-cpu  flashrank  sentence-transformers langchain-text-splitters langchain-groq langchain-community langchain-core  langchain-core tqdm  nltk pydantic rank_bm25 langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137

**Imports**

In [ ]:
import os
import json
import re
import time
import numpy as np
import pandas as pd
import itertools
from typing import List, Dict, Any, Tuple

# Core Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
import nltk
from nltk.tokenize import sent_tokenize

# Core Retrieval
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_community.document_compressors import FlashrankRerank

# Vector DB & Groq
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from rank_bm25 import BM25Okapi
from flashrank import Ranker, RerankRequest


from google.colab import userdata
from google.colab import drive
import shutil

/tmp/ipykernel_2297/2736742739.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_compressors import FlashrankRerank


**Util Function**

In [ ]:
#Convert Data Frame to lang Chain docs

def convert_dataframe_to_langchain_docs(df: pd.DataFrame) -> List[Document]:
    """
    Converts a RAGBench style pandas DataFrame row into a flat list of
    standard LangChain Document objects with structured metadata.
    """
    langchain_documents = []
    #print(df.head())

    for _, row in df.iterrows():
        # Extract metadata fields from the row
        row_id = str(row.get('id', 'N/A'))
        dataset_name = str(row.get('dataset_name', 'unknown'))

        # 'documents' column contains a list of string passages
        passages = row.get('documents', [])

        #converting into single string
        finalPassage = ["\n".join(passages)]

        if isinstance(finalPassage, list):
            for idx, passage_text in enumerate(finalPassage):
              # Build metadata dictionary unique to each passage
              metadata = {
                  "id": row_id,
                  "dataset_name": dataset_name,
                  "passage_index": idx  # Tracks original position inside the row array
              }
              # Instantiate standard LangChain Document object
              doc = Document(
                  page_content=passage_text,
                  metadata=metadata
              )
              langchain_documents.append(doc)

    print(f"✅ Successfully converted DataFrame rows into {len(langchain_documents)} LangChain Documents.")
    return langchain_documents


#split the doc and answer into sentences
def simple_sentence_split(text):
    text = text.replace("\n", " ").strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    return sentences


#Generate keys for each sentence in the corpus
def key_context_sentences(retrieved_docs: List[Document]):
    keyed_sentences = []
    context_text = ""

    letters = "abcdefghijklmnopqrstuvwxyz"

    for doc_idx, doc in enumerate(retrieved_docs):
        sentences = simple_sentence_split(doc.page_content)

        context_text += f"\nDocument {doc_idx}:\n"

        for sent_idx, sent in enumerate(sentences):
            if sent_idx < len(letters):
                key = f"{doc_idx}{letters[sent_idx]}"
            else:
                key = f"{doc_idx}{sent_idx}"

            keyed_sentences.append({
                "key": key,
                "sentence": sent
            })

            context_text += f"{key}: {sent}\n"

    return keyed_sentences, context_text


#Generate keys for each sentence in the LLM answer
def key_response_sentences(response_text):
    sentences = simple_sentence_split(response_text)
    letters = "abcdefghijklmnopqrstuvwxyz"

    keyed_response = []
    response_text_keyed = ""

    for i, sent in enumerate(sentences):
        key = letters[i] if i < len(letters) else str(i)
        keyed_response.append({
            "key": key,
            "sentence": sent
        })
        response_text_keyed += f"{key}: {sent}\n"

    return keyed_response, response_text_keyed


#parse JSON returned by Judge LLM

def parse_json_safely(text):
    try:
        return json.loads(text)
    except:
        pass

    # Try extracting JSON object from text
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            return None

    return None



#Method to detect rate limit err
def is_rate_limit_error(error):
    error_text = str(error).lower()
    rate_limit_patterns = [
        "rate limit",
        "rate_limit_exceeded",
        "tokens per day",
        "tpd",
        "429"
    ]
    return any(pattern in error_text for pattern in rate_limit_patterns)

#Checkpoint Timelap Verification
def isCheckPointRequired(last_save_time, duration):
  current_time = time.time()
  elapsed_since_save = current_time - last_save_time
  if elapsed_since_save >= duration:
    return True, current_time
  else:
    return False, last_save_time

#Move local Data To Drive
def moveRecordsToDrive():
  results_df = pd.read_csv(CHECKPOINT_FILE)

  successful_rows = results_df[results_df["error"].isna()]

  print("Successful rows:", successful_rows.shape[0])
  print("Total test rows:", len(test_data))

  if successful_rows.shape[0] == len(test_data):
    successful_rows.to_csv(FINAL_FILE, index=False)
    shutil.copy(
      FINAL_FILE,
      FINAL_DRIVE_FILE
    )
    print("Full File copied to Google Drive.")
    print("Full test completed.")
    print("Final file saved:", FINAL_FILE)
  else:
    shutil.copy(
        CHECKPOINT_FILE,
        CHECKPOINT_DRIVE_FILE
    )
    print("Checkpoint copied to Google Drive.")
    print("Full test is not complete yet.")
    print("Resume later from checkpoint.")

**Metrics Computation**

In [ ]:
#Compute TRACe metrics

def compute_trace_metrics(judge_output, total_context_sentence_count):
    relevant_keys = set(judge_output.get("all_relevant_sentence_keys", []))
    utilized_keys = set(judge_output.get("all_utilized_sentence_keys", []))

    # Context Relevance
    if total_context_sentence_count == 0:
        relevance_score = 0.0
    else:
        relevance_score = len(relevant_keys) / total_context_sentence_count

    # Context Utilization
    if total_context_sentence_count == 0:
        utilization_score = 0.0
    else:
        utilization_score = len(utilized_keys) / total_context_sentence_count

    # Completeness
    if len(relevant_keys) == 0:
        completeness_score = None
    else:
        completeness_score = len(relevant_keys.intersection(utilized_keys)) / len(relevant_keys)

    # Adherence
    adherence_score = True if judge_output.get("overall_supported", False) else False

    support_info = judge_output.get("sentence_support_information", [])
    if support_info:
        supported = sum(1 for s in support_info if s.get("fully_supported", False))
        adherence_continuous = supported / len(support_info)
    else:
        adherence_continuous = 1 if adherence_score else 0


    return {
        "pred_relevance_score": relevance_score,
        "pred_utilization_score": utilization_score,
        "pred_completeness_score": completeness_score,
        "pred_adherence_score": adherence_score,
        "pred_adherence_continuous": adherence_continuous
    }

**Chuncking**

In [ ]:
nltk.download("punkt")
nltk.download('punkt_tab')

class SentenceTextSplitter:
    """Sentence-boundary-aware splitter. Document in, Document out."""

    def __init__(self, sentences_per_chunk: int = 1, sentence_overlap: int = 0):
        self.sentences_per_chunk = sentences_per_chunk
        self.sentence_overlap = sentence_overlap

    def split_text(self, text: str) -> List[str]:
        sentences = sent_tokenize(text)
        step = max(self.sentences_per_chunk - self.sentence_overlap, 1)
        chunks = []
        for i in range(0, len(sentences), step):
            group = sentences[i:i + self.sentences_per_chunk]
            if not group:
                continue
            chunks.append(" ".join(group))
            if i + self.sentences_per_chunk >= len(sentences):
                break
        return chunks

    def split_documents(self, documents: List[Document]) -> List[Document]:
        output_docs = []
        for doc in documents:
            text_chunks = self.split_text(doc.page_content)
            for idx, chunk_text in enumerate(text_chunks):
                new_metadata = dict(doc.metadata)  # copy parent metadata
                new_metadata.update({
                    "chunk_sentence_index": idx,
                    "chunk_strategy": "sentence_level",
                })
                output_docs.append(Document(page_content=chunk_text, metadata=new_metadata))
        return output_docs

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
#Hierarchical
import uuid

class HierarchicalChuncking:
    def __init__(self, chunck_type:str, parent_chunck_multiplier: int = 3, child_chunk_size: int = 400, overlap_size: int = 80):
      self.parent_chuck_size = child_chunk_size * parent_chunck_multiplier
      self.child_chunk_size = child_chunk_size
      self.overlap_size = overlap_size
      self.chunck_type = chunck_type

    def split_documents(self, documents: List[Document]):
        output_docs = []
        parent_store = {}      # parent_id -> parent text

        if self.chunck_type == "charsplit":
          parent_splitter = RecursiveCharacterTextSplitter(chunk_size=self.parent_chuck_size, chunk_overlap=self.overlap_size)
          child_splitter = RecursiveCharacterTextSplitter(chunk_size=self.child_chunk_size, chunk_overlap=self.overlap_size)

        elif self.chunck_type == "sentence":
          parent_splitter = SentenceTextSplitter(self.parent_chuck_size, self.overlap_size)
          child_splitter = SentenceTextSplitter(self.child_chunk_size, self.overlap_size)

        for doc in documents:
            parents = parent_splitter.split_text(doc.page_content)

            for parent_text in parents:
                parent_id = str(uuid.uuid4())
                parent_store[parent_id] = Document(
                  page_content=parent_text,
                  metadata=doc.metadata
                )
                children = child_splitter.split_text(parent_text)

                for child_text in children:
                    new_metadata = dict(doc.metadata)  # copy parent metadata
                    new_metadata.update({
                        "parent_id": parent_id,
                        "chunk_strategy": "hierarchical"
                    })
                    output_docs.append(Document(page_content=child_text, metadata=new_metadata))

        return output_docs, parent_store


**Vector Store For load and Retrieve**

In [ ]:
# ── Embeddings ────────────────────────────────────────────────
class HuggingFaceEmbedder:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, convert_to_numpy=True)

    def embed_query(self, text: str) -> np.ndarray:
        return self.model.encode([text], convert_to_numpy=True)[0]

In [ ]:
class HybridRetriever:
    """
    Hybrid retrieval combining dense and sparse methods using Reciprocal Rank Fusion.
    """

    def __init__(self, documents: List[Document], embedder: HuggingFaceEmbedder, alpha: float = 0.5):
        """
        Initialize hybrid retriever with both dense and sparse retrievers.

        Args:
            documents: List of LangChain Document objects
            embedder: HuggingFaceEmbedder instance
            alpha: Weight for combining scores (0=sparse only, 1=dense only, 0.5=balanced)
        """
        self.documents = documents
        self.alpha = alpha

        print(f"Initializing hybrid retriever (alpha={alpha})...")

        # Initialize both retrievers
        print("  - Initializing sparse (BM25) retriever...")
        self.bm25_retriever = BM25Retriever(documents)

        print("  - Initializing dense (vector) retriever...")
        self.vector_retriever = VectorRetriever(documents, embedder)

        print("✅ Hybrid retriever initialized")

    def _reciprocal_rank_fusion(
        self,
        bm25_results: List[Tuple[int, float]],
        vector_results: List[Tuple[int, float]],
        k: int = 60
    ) -> List[int]:
        """
        Combine results using Reciprocal Rank Fusion (RRF).

        Args:
            bm25_results: List of (doc_index, score) from BM25
            vector_results: List of (doc_index, score) from vector search
            k: Constant for RRF (default 60)

        Returns:
            List of document indices sorted by combined score
        """
        # Calculate RRF scores
        rrf_scores: Dict[int, float] = {}

        # Add BM25 scores with weight (1-alpha)
        for rank, (doc_idx, _) in enumerate(bm25_results, start=1):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + (1 - self.alpha) / (k + rank)

        # Add vector scores with weight alpha
        for rank, (doc_idx, _) in enumerate(vector_results, start=1):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + self.alpha / (k + rank)

        # Sort by combined score
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_idx for doc_idx, _ in sorted_docs]

    def invoke(self, query: str, k: int = 3) -> List[Document]:
        """
        Retrieve top-k documents using hybrid approach.

        Args:
            query: Query string
            k: Number of documents to retrieve

        Returns:
            List of top-k Document objects
        """
        # Retrieve more docs from each method for better fusion
        retrieve_k = k * 2

        # Get BM25 results with scores
        bm25_scores = self.bm25_retriever.bm25.get_scores(
            self.bm25_retriever._tokenize(query)
        )
        bm25_top_indices = np.argsort(bm25_scores)[::-1][:retrieve_k]
        bm25_results = [(idx, bm25_scores[idx]) for idx in bm25_top_indices if bm25_scores[idx] > 0]

        # Get vector results with scores
        query_embedding = self.vector_retriever.embedder.embed_query(query)
        vector_scores = self.vector_retriever._cosine_similarity(
            query_embedding,
            self.vector_retriever.doc_embeddings
        )
        vector_top_indices = np.argsort(vector_scores)[::-1][:retrieve_k]
        vector_results = [(idx, vector_scores[idx]) for idx in vector_top_indices]

        # Combine using RRF
        combined_indices = self._reciprocal_rank_fusion(bm25_results, vector_results)

        # Return top-k documents
        return [self.documents[idx] for idx in combined_indices[:k]]

# ── BM25 Retriever ────────────────────────────────────────────
class BM25Retriever:
    def __init__(self, documents: List[Document]):
        self.documents = documents
        self.bm25 = self._build_index()

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().split()

    def _build_index(self) -> BM25Okapi:
        tokenized = [self._tokenize(doc.page_content) for doc in self.documents]
        return BM25Okapi(tokenized)

    def invoke(self, query: str, k: int = 3) -> List[Document]:   # ✅ k at retrieval time
        scores = self.bm25.get_scores(self._tokenize(query))
        top_k = np.argsort(scores)[::-1][:k]
        return [self.documents[i] for i in top_k if scores[i] > 0]


# ── Vector Retriever ──────────────────────────────────────────
class VectorRetriever:
    def __init__(self, documents: List[Document], embedder: HuggingFaceEmbedder):
        self.documents = documents
        self.embedder = embedder
        self.doc_embeddings = self._build_index()

    def _build_index(self) -> np.ndarray:
        texts = [doc.page_content for doc in self.documents]
        return self.embedder.embed_documents(texts)

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        norms = np.linalg.norm(b, axis=1) * np.linalg.norm(a)
        return np.dot(b, a) / (norms + 1e-9)

    def invoke(self, query: str, k: int = 3) -> List[Document]:   # ✅ k at retrieval time
        query_embedding = self.embedder.embed_query(query)
        scores = self._cosine_similarity(query_embedding, self.doc_embeddings)
        top_k = np.argsort(scores)[::-1][:k]
        return [self.documents[i] for i in top_k]


# ── Main Class ────────────────────────────────────────────────
class VectorDBAndEmbeddingAlongRetriever:
    def __init__(self, raw_documents: List[Document], config: Dict[str, Any]):
        self.embedder = HuggingFaceEmbedder(config["embedding_model"])
        self.vector_retriever = None
        self.parent_store = None
        print("\nData Chunking Started......")
        if config["chunk_type"] == "charsplit":
          text_splitter = RecursiveCharacterTextSplitter(
              chunk_size=config["chunk_size"],
              chunk_overlap=config["chunk_overlap"],
              length_function=len
            )
          self.chunks = text_splitter.split_documents(raw_documents)
        elif config["chunk_type"] == "hierarchical":
          text_splitter = HierarchicalChuncking(config["sub_chunk_type"], config["hierarchical_parent_group"], config["chunk_size"], config["chunk_overlap"])
          res_chunks, parent_store = text_splitter.split_documents(raw_documents)
          self.chunks = res_chunks
          self.parent_store = parent_store
        else:
          self.chunks = SentenceTextSplitter(config["chunk_size"], config["chunk_overlap"]).split_documents(raw_documents)

        print("\nData Chunking Finished......")

    def _get_vector_store_as_retriever(
        self) -> VectorRetriever:
        print("\n Document Embedding started for dense retriever_type ......")
        return VectorRetriever(documents=self.chunks, embedder=self.embedder)

    def load_documents(self, config: Dict[str, Any]):
        retriever_type = config["retriever_type"].lower()

        if retriever_type == "dense":
            self.vector_retriever = self._get_vector_store_as_retriever()
            print("\n Document Embedding Finished for dense retriever_type ......")

        elif retriever_type == "sparse":
            print("\n Document Embedding started for sparse retriever_type......")
            self.vector_retriever = BM25Retriever(documents=self.chunks)
            print("\n Document Embedding Finished for sparse retriever_type......")

        elif retriever_type == "hybrid":
            print("\n Document Embedding started for hybrid retriever_type......")
            self.vector_retriever = HybridRetriever(documents=self.chunks, embedder=self.embedder, alpha = 0.5)
            print("\n Document Embedding Finished for hybrid retriever_type......")

        else:
            raise ValueError(f"Retriever type '{config['retriever_type']}' is not supported.")

    def get_related_docs(self, query: str, k: int = 3) -> List[Document]:  # ✅ k passed here
        docs = []
        if self.vector_retriever is None:
            raise RuntimeError("Call load_documents() before get_related_docs()")

        if config["chunk_type"] == "hierarchical":

          child_docs = self.vector_retriever.invoke(query, k=k)
          seen, parents = set(), []
          for child_doc in child_docs:
              pid = child_doc.metadata["parent_id"]
              if pid not in seen:
                  seen.add(pid)
                  parents.append(self.parent_store[pid])          # swap in full parent
          docs = parents

        else:
          docs = self.vector_retriever.invoke(query, k=k)  # ✅ k forwarded to retriever
        return docs


**Re Ranking Documents Along Repacking**

In [ ]:
# Re Ranking and Repacking The documents

class RankAndRepackDocuments:
  def __init__(self, config: Dict[str, Any]):
    # Initialize engine inline or globally as per notebook structure
    self.rank_engine = Ranker(model_name=config["ranking_model"], cache_dir="/tmp")

  def get_documents(self, retrieved_docs:List[Document], query: str, config: Dict[str, Any]):

    # Format docs into FlashRank payloads
    passages = [
        {"id": idx, "text": doc.page_content, "meta": doc.metadata}
        for idx, doc in enumerate(retrieved_docs)
    ]

    rank_request = RerankRequest(query=query, passages=passages)
    print(rank_request)
    ranked_results = self.rank_engine.rerank(rank_request)

    # Reconstruct LangChain Documents according to new ranks
    rerank_k = config.get("rerank_k", 3)
    reranked_docs = [
        Document(page_content=r["text"], metadata=r["meta"])
        for r in ranked_results[:rerank_k]
    ]

    if config.get("repacking_strategy", "sides") == "sides":
      repacked_docs = [None] * len(reranked_docs)
      left, right = 0, len(reranked_docs) - 1
      for i, doc in enumerate(reranked_docs):
          if i % 2 == 0:
              repacked_docs[left] = doc
              left += 1
          else:
              repacked_docs[right] = doc
              right -= 1
      return [d for d in repacked_docs if d is not None]
    else:
        return reranked_docs[::-1]


**RAG Main Pipeline**

In [ ]:
# RAG Pipeline

class RAGExperimentPipeline:
  def __init__(self, raw_documents: List[Document], config: Dict[str, Any]):
    # Define vector store and retreiver type based on config
    self.data_store = VectorDBAndEmbeddingAlongRetriever(raw_documents, config)
    print("\n Intilizing Data Store Done")

    # Load raw documents to vector store
    print("\n Loading Data into Data Store Started")
    self.data_store.load_documents(config)
    print("\n Loading Data into Data Store Done")

    # Initilize RaRank and Repacking
    self.re_rank_and_repack = RankAndRepackDocuments(config)
    print("\n Intilizing Re Rank and Repacking are Done")

  def retrerive_docs(self, query: str, config: Dict[str, Any]):
    # Get releated documents
    related_docs = self.data_store.get_related_docs(query, config["top_k"])
    #print("related docs::  \n")
    #print(related_docs)

    # Re rank and repack the related docs
    reranked_docs = self.re_rank_and_repack.get_documents(related_docs, query, config)
    #print("re ranked docs::  \n")
    #print(reranked_docs)
    return reranked_docs


  def run_answer_generation(self, query: str, context_docs: List[Document], llm: ChatGroq, config: Dict[str, Any]) -> str:
    # Generation Step via Groq
    prompt_template = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template("You are a helpful assistant."),
        HumanMessagePromptTemplate.from_template("""
          You are a highly precise engineering assistant. Answer the question based strictly on the provided context.
          If the context does not contain the answer, perform negative rejection and say you do not know.

          Context:
          {context}

          Question: {question}
          Answer:
          """)
    ])

    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({"context": context_docs, "question": query})
    #print(response)
    return response

  def run_judge_llm(self, query: str, documents: str, generated_reponse: str, llm: ChatGroq, config: Dict[str, Any]):
      prompt_template = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template("You are a strict RAG evaluation judge. Return only valid JSON."),
        HumanMessagePromptTemplate.from_template("""
            You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems. Your task is to review a response provided for a given question based on one or more source documents and evaluate alignment, relevance, and hallucination sentence by sentence.

            Here are the source documents, split into sentences with unique keys (e.g., '0a.', '0b.'):
            {context_documents}

            The user question was:
            {user_question}

            Here is the provided response, split into sentences with unique keys (e.g., 'a.', 'b.'):
            {llm_generated_answer}

            Evaluate the inputs and output a single, valid JSON object matching this exact schema. Do not include markdown code blocks (```json), backticks, or any conversational text before or after the JSON string. Escape all nested quotes (\") and newlines (\n).

            {{
              "relevance_explanation": "string",
              "all_relevant_sentence_keys": ["string"],
              "overall_supported_explanation": "string",
              "overall_supported": boolean,
              "sentence_support_information": [
                {{
                  "response_sentence_key": "string",
                  "explanation": "string",
                  "supporting_sentence_keys": ["string"],
                  "fully_supported": boolean
                }}
              ],
              "all_utilized_sentence_keys": ["string"]
            }}

            ---
            FIELD DEFINITIONS & STRICT LOGIC RULES:

            1. "relevance_explanation"
            - Content: A step-by-step breakdown explaining which source documents contain useful information for answering the question and exactly how that information is helpful.

            2. "all_relevant_sentence_keys"
            - Content: An array of all document sentence keys relevant to the question.
            - Rule: Include every sentence useful to the question, even if it was completely omitted from the provided response, or if only a portion of it is useful. Base this judgment strictly on the source documents and the question; ignore the provided response entirely. Omit sentences that could be deleted without impacting a human's ability to answer the question.

            3. "overall_supported_explanation"
            - Content: A step-by-step evaluation of why the response as a whole is or is not supported by the documents.
            - Constraint: You MUST evaluate each response claim separately, one by one, in isolation first. Do not make any summary remarks about the response as a whole until all isolated claims have been completely assessed.

            4. "overall_supported"
            - Content: Boolean (true/false) indicating if the entire response is supported. This must logically match the final conclusion drawn in "overall_supported_explanation".

            5. "sentence_support_information"
            - Content: A list containing exactly one object per sentence in the provided response.
              * "response_sentence_key": Matches the key of the sentence from the provided response.
              * "explanation": A detailed string explaining why this specific sentence is or is not supported by the source text.
              * "supporting_sentence_keys": Array of keys from the source documents that support this specific response sentence.
                - If the sentence is NOT supported, this array MUST be empty.
                - If the sentence IS supported, provide the source keys.
                - Special Case Exceptions: If a sentence is supported but has no specific source key, populate this field with one of these literal string identifiers instead:
                  - "supported_without_sentence": If the response sentence expresses an inability to answer due to missing context information, or is supported generally by the collective text.
                  - "general": For transition sentences, summaries of previous sentences, or outlines of the answering steps.
                  - "well_known_fact": If the sentence states a universally known fact (e.g., a math formula).
                  - "numerical_reasoning": If the sentence executes basic math logic (e.g., addition, multiplication).
              * "fully_supported": Boolean (true/false).
                - If "supporting_sentence_keys" is empty, this MUST be false.
                - Set to true only if every single claim within the response sentence is perfectly covered by the keys in "supporting_sentence_keys". Set to false if it is only partially or incompletely supported.

            6. "all_utilized_sentence_keys"
            - Content: An array of all source document sentence keys that were actively used to construct the response.
            - Rule: Include keys that directly supported the answer or were implicitly used to build it (even if the source sentence was not used in its entirety). Omit source keys that were completely ignored or had no bearing on the final answer.
            """)
      ])

      chain = prompt_template | llm | StrOutputParser()
      response = chain.invoke({
          "context_documents": documents,
          "user_question": query,
          "llm_generated_answer": generated_reponse
      })
      return response



**Pipelien Execution Step**

In [ ]:
#Config Loading
config = {
    "data_set" : "msmarco",
    "data_type": "test",
    "chunk_type" : "hierarchical", # or ("charsplit" , "hierarchical", "sentence")
    "sub_chunk_type" : "sentence", # or ("charsplit" , "sentence")
    "hierarchical_parent_group" : 5,
    "chunk_size" : 3,
    "chunk_overlap" : 1,
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "retriever_type": "dense",  # or ("sparse", "dense", "hybrid")
    "top_k" : 6,
    "ranking_model": "ms-marco-MiniLM-L-12-v2",
    "rerank_k" : 4,
    "repacking_strategy" : "sides",
    "domain" : "gk",
    "generator_model" : "google/gemini-3.5-flash-lite",
    "judge_model" : "meta-llama/llama-3.3-70b-instruct",
    "llm_provider" : "gemini",
    "genai_provider" : "groq1"
}

In [ ]:
# Constant File name definition

FILE_FORMAT = f"{config["data_set"]}_{config["chunk_type"]}_{config["sub_chunk_type"]}_{config["hierarchical_parent_group"]}_{config["chunk_size"]}_{config["chunk_overlap"]}_{config["retriever_type"]}_{config["top_k"]}_{config["rerank_k"]}_{config["repacking_strategy"]}"
CHECKPOINT_FILE = f"ragbench_{config["llm_provider"]}_{FILE_FORMAT}_checkpoint.csv"
FINAL_FILE = f"ragbench_{config["llm_provider"]}_{FILE_FORMAT}_final.csv"

drive.mount('/content/drive')
CHECKPOINT_DRIVE_FILE = f"/content/drive/MyDrive/RAG_Project/{CHECKPOINT_FILE}"
FINAL_DRIVE_FILE = f"/content/drive/MyDrive/RAG_Project/{FINAL_FILE}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Secrets handling
secret_keys = []. # Added groq secret keys

secret_keys_index = 0
os.environ["GROQ_API_KEY"] = secret_keys[secret_keys_index]

secret_keys_open_router = [] # add open router secret keys

secret_keys_open_router_index = 0
os.environ["OPENROUTER_API_KEY"] = secret_keys_open_router[secret_keys_open_router_index]

In [ ]:
# Data Set handling Part
from datasets import load_dataset

dataset = load_dataset("rungalileo/ragbench",config["data_set"])
test_data = dataset[config["data_type"]].select(range(50))

df = test_data.to_pandas()
#df.head()

**Pre steps before Execution**

In [ ]:
# Data Frame to docs
raw_documents = convert_dataframe_to_langchain_docs(df)


✅ Successfully converted DataFrame rows into 50 LangChain Documents.


In [ ]:
# Rag pipeline
rag_pipeline = RAGExperimentPipeline(raw_documents, config)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Data Chunking Started......

Data Chunking Finished......

 Intilizing Data Store Done

 Loading Data into Data Store Started

 Document Embedding started for dense retriever_type ......

 Document Embedding Finished for dense retriever_type ......

 Loading Data into Data Store Done

 Intilizing Re Rank and Repacking are Done


In [ ]:
secret_keys_index = 0
os.environ["GROQ_API_KEY"] = secret_keys[secret_keys_index]
secret_keys_open_router_index = 0
os.environ["OPENROUTER_API_KEY"] = secret_keys_open_router[secret_keys_open_router_index]

In [ ]:
def isCheckPointRequired(last_save_time, duration):
  current_time = time.time()
  elapsed_since_save = current_time - last_save_time
  if elapsed_since_save >= duration:
    return True, current_time
  else:
    return False, last_save_time

In [ ]:
def moveRecordsToDrive():
  results_df = pd.read_csv(CHECKPOINT_FILE)

  successful_rows = results_df[results_df["error"].isna()]

  print("Successful rows:", successful_rows.shape[0])
  print("Total test rows:", len(test_data))

  if successful_rows.shape[0] == len(test_data):
    successful_rows.to_csv(FINAL_FILE, index=False)
    shutil.copy(
      FINAL_FILE,
      FINAL_DRIVE_FILE
    )
    print("Full File copied to Google Drive.")
    print("Full test completed.")
    print("Final file saved:", FINAL_FILE)
  else:
    shutil.copy(
        CHECKPOINT_FILE,
        CHECKPOINT_DRIVE_FILE
    )
    print("Checkpoint copied to Google Drive.")
    print("Full test is not complete yet.")
    print("Resume later from checkpoint.")

**RAG Execution**

In [ ]:

import time
from datetime import datetime
save_interval = 300

# Copy Files from Drive to local
try:
  shutil.copy(
        CHECKPOINT_DRIVE_FILE,
        CHECKPOINT_FILE,
    )
  print("\n Checkpoint copied From Drive to local")
except Exception as e:
  print("\n No Checkpoint file to copy")

try:
  shutil.copy(
      FINAL_DRIVE_FILE,
      FINAL_FILE
  )
  print("\n Final file copied From Drive to local")
except Exception as e:
  print("\n No Final file to copy")

# Check existing computation results
if os.path.exists(CHECKPOINT_FILE):
    results_df_existing = pd.read_csv(CHECKPOINT_FILE)

    if "row_id" in results_df_existing.columns:
        completed_ids = set(
            results_df_existing[
                results_df_existing["error"].isna()
            ]["row_id"].astype(int).tolist()
        )
    else:
        completed_ids = set()

    results = results_df_existing.to_dict("records")

    print("Existing checkpoint found.")
    print("Successful completed rows:", len(completed_ids))
    print("Total saved rows in checkpoint:", len(results))
else:
    completed_ids = set()
    results = []

    print("No checkpoint found. Starting fresh.")


last_save_time = time.time()
isChekpointRequired = None
# Execute the pipeline for records
for i in tqdm(range(len(test_data)), desc="Running/resuming full test evaluation"):

    # Skip rows already completed successfully
    if i in completed_ids:
        continue

    row = test_data[i]
    question = row["question"]

    try:
        # 1. Retrieve (fecth, re rank and reapck)
        retrieved_docs = rag_pipeline.retrerive_docs(question, config)

        if config.get("genai_provider", "groq") == "groq":
          answer_llm = ChatGroq(model=config["generator_model"], temperature=0)
          judge_llm = ChatGroq(model=config["judge_model"], temperature=0)
        else:
          answer_llm = ChatOpenAI(
            openai_api_base="https://openrouter.ai/api/v1",
            openai_api_key=os.environ["OPENROUTER_API_KEY"],
            model_name=config["generator_model"],
            temperature=0.0,  # Keep at 0.0 for consistent and precise factual extraction in RAG
            default_headers={
                "HTTP-Referer": "http://localhost:3000", # Optional: Your site URL for OpenRouter ranking analytics
                "X-Title": "AIML RAG Project"            # Optional: Your application name
            }
          )

          judge_llm = ChatOpenAI(
            openai_api_base="https://openrouter.ai/api/v1",
            openai_api_key=os.environ["OPENROUTER_API_KEY"],
            model_name="meta-llama/llama-3.3-70b-instruct",
            temperature=0.0,  # Keep at 0.0 for consistent and precise factual extraction in RAG
            default_headers={
                "HTTP-Referer": "http://localhost:3000", # Optional: Your site URL for OpenRouter ranking analytics
                "X-Title": "AIML RAG Project"            # Optional: Your application name
            }
          )

        # 2. Generate answer using Groq
        generated_response = rag_pipeline.run_answer_generation(question, retrieved_docs, answer_llm, config)

        # 3. Key context and response
        keyed_context, context_for_judge = key_context_sentences(retrieved_docs)
        keyed_response, response_for_judge = key_response_sentences(generated_response)

        # 4. Judge using Groq
        judge_response = rag_pipeline.run_judge_llm(question, context_for_judge, response_for_judge, judge_llm, config)

        judge_stats = parse_json_safely(judge_response)

        if judge_stats is None:
            raise ValueError("Judge output could not be parsed as JSON")

        # 5. Compute metrics
        total_context_sentence_count = len(keyed_context)
        pred_metrics = compute_trace_metrics(judge_stats,total_context_sentence_count)

        # 6. Store successful result
        result = {
            "row_id": i,
            "question": question,
            "generated_response": generated_response,
            "retrieved_doc_ids": json.dumps([doc.metadata["id"] for doc in retrieved_docs]),
            "judge_output": json.dumps(judge_stats),
            "total_context_sentence_count": total_context_sentence_count,

            "pred_relevance_score": pred_metrics["pred_relevance_score"],
            "pred_utilization_score": pred_metrics["pred_utilization_score"],
            "pred_completeness_score": pred_metrics["pred_completeness_score"],
            "pred_adherence_score": pred_metrics["pred_adherence_score"],
            "pred_adherence_continuous": pred_metrics["pred_adherence_continuous"],

            "gold_relevance_score": row.get("relevance_score", None),
            "gold_utilization_score": row.get("utilization_score", None),
            "gold_completeness_score": row.get("completeness_score", None),
            "gold_adherence_score": row.get("adherence_score", None),

            "error": None
        }

        results.append(result)
        completed_ids.add(i)

        # Save immediately after every successful row
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)

        print(f"Saved row {i}. Total completed: {len(completed_ids)} / {len(test_data)}")
        isChekpointRequired, last_save_time = isCheckPointRequired(last_save_time, save_interval)
        if isChekpointRequired:
          moveRecordsToDrive()

        if config.get("genai_provider", "groq") == "groq":
          time.sleep(1)

    except Exception as e:
        # If Groq token/rate limit happens, save progress and stop safely
        if is_rate_limit_error(e):
            print("\nGroq token/rate limit reached.")
            print("Error message:")
            print(e)
            if config.get("genai_provider", "groq") == "groq" and len(secret_keys) > secret_keys_index+1 :
              secret_keys_index += 1
              os.environ["GROQ_API_KEY"] = secret_keys[secret_keys_index]
              continue
            elif config.get("genai_provider", "groq") != "groq" and len(secret_keys_open_router) > secret_keys_open_router_index+1 :
              secret_keys_open_router_index += 1
              os.environ["OPENROUTER_API_KEY"] = secret_keys_open_router[secret_keys_open_router_index]
              continue
            else:
              print("\nNo keys left to try, stopping the process")
              print("\nSaving checkpoint and stopping safely.")

              pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
              isChekpointRequired, last_save_time = isCheckPointRequired(last_save_time, save_interval)
              if isChekpointRequired:
                moveRecordsToDrive()

              print("Checkpoint saved:", CHECKPOINT_FILE)
              print("Completed rows:", len(completed_ids), "/", len(test_data))
              print("Next row to resume from:", i)

              break

        # For non-rate-limit errors, save the row as failed and continue
        else:
            print(f"\nError on row {i}: {e}")

            error_result = {
                "row_id": i,
                "question": question,
                "generated_response": None,
                "retrieved_doc_ids": None,
                "retrieved_doc_scores": None,
                "judge_output": None,
                "total_context_sentence_count": None,

                "pred_relevance_score": None,
                "pred_utilization_score": None,
                "pred_completeness_score": None,
                "pred_adherence_score": None,
                "pred_adherence_continuous": None,

                "gold_relevance_score": row.get("relevance_score", None),
                "gold_utilization_score": row.get("utilization_score", None),
                "gold_completeness_score": row.get("completeness_score", None),
                "gold_adherence_score": row.get("adherence_score", None),

                "error": str(e)
            }

            results.append(error_result)

            # Save after error too
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
            isChekpointRequired, last_save_time = isCheckPointRequired(last_save_time, save_interval)
            if isChekpointRequired:
              moveRecordsToDrive()

            continue

moveRecordsToDrive()
print("\nRun finished or stopped safely.")
print("Checkpoint file:", CHECKPOINT_FILE)


 Checkpoint copied From Drive to local

 No Final file to copy
Existing checkpoint found.
Successful completed rows: 49
Total saved rows in checkpoint: 50


Running/resuming full test evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

Running/resuming full test evaluation: 100%|██████████| 50/50 [00:14<00:00,  3.36it/s]

Saved row 34. Total completed: 50 / 50
Successful rows: 50
Total test rows: 50
Full File copied to Google Drive.
Full test completed.
Final file saved: ragbench_gemini_msmarco_hierarchical_sentence_5_3_1_dense_6_4_sides_final.csv

Run finished or stopped safely.
Checkpoint file: ragbench_gemini_msmarco_hierarchical_sentence_5_3_1_dense_6_4_sides_checkpoint.csv


**Move Results to Drive**

In [ ]:
#Move Results Drive
results_df = pd.read_csv(CHECKPOINT_FILE)

successful_rows = results_df[results_df["error"].isna()]

print("Successful rows:", successful_rows.shape[0])
print("Total test rows:", len(test_data))

if successful_rows.shape[0] == len(test_data):
  successful_rows.to_csv(FINAL_FILE, index=False)
  shutil.copy(
    FINAL_FILE,
    FINAL_DRIVE_FILE
  )
  print("Full File copied to Google Drive.")
  print("Full test completed.")
  print("Final file saved:", FINAL_FILE)
else:
  shutil.copy(
      CHECKPOINT_FILE,
      CHECKPOINT_DRIVE_FILE
  )
  print("Checkpoint copied to Google Drive.")
  print("Full test is not complete yet.")
  print("Resume later from checkpoint.")

Successful rows: 50
Total test rows: 50
Full File copied to Google Drive.
Full test completed.
Final file saved: ragbench_gemini_msmarco_hierarchical_sentence_5_3_1_dense_6_4_sides_final.csv


In [ ]:
config

{'data_set': 'msmarco',
 'data_type': 'test',
 'chunk_type': 'hierarchical',
 'sub_chunk_type': 'sentence',
 'hierarchical_parent_group': 5,
 'chunk_size': 3,
 'chunk_overlap': 1,
 'embedding_model': 'BAAI/bge-small-en-v1.5',
 'retriever_type': 'dense',
 'top_k': 6,
 'ranking_model': 'ms-marco-MiniLM-L-12-v2',
 'rerank_k': 4,
 'repacking_strategy': 'sides',
 'domain': 'gk',
 'generator_model': 'google/gemini-3.5-flash-lite',
 'judge_model': 'meta-llama/llama-3.3-70b-instruct',
 'llm_provider': 'gemini',
 'genai_provider': 'groq1'}

In [ ]:
#Model Performance
SUMMARY_FILE = f"ragbench_{config["llm_provider"]}_{FILE_FORMAT}_results_summary.json"
SUMMARY_DRIVE_FILE = f"/content/drive/MyDrive/RAG_Project/{SUMMARY_FILE}"


from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

results_df = pd.read_csv(FINAL_FILE)
results_df['pred_completeness_score'] = results_df['pred_completeness_score'].fillna(0)

metrics_summary = {}

metrics_summary["data_set"] = config["data_set"]
metrics_summary["data_type"] = config["data_type"]
metrics_summary["chunk_type"] = config["chunk_type"]
metrics_summary["sub_chunk_type"] = config["sub_chunk_type"]
metrics_summary["chunk_size"] = config["chunk_size"]
metrics_summary["chunk_overlap"] = config["chunk_overlap"]
metrics_summary["embedding_model"] = config["embedding_model"]
metrics_summary["retriever_type"] = config["retriever_type"]
metrics_summary["top_k"] = config["top_k"]
metrics_summary["ranking_model"] = config["ranking_model"]
metrics_summary["rerank_k"] = config["rerank_k"]
metrics_summary["repacking_strategy"] = config["repacking_strategy"]
metrics_summary["generator_model"] = config["generator_model"]
metrics_summary["judge_model"] = config["judge_model"]

metrics_summary["mse_relevance"] = mean_squared_error(results_df["gold_relevance_score"], results_df["pred_relevance_score"])
metrics_summary["rmse_relevance"] = root_mean_squared_error(results_df["gold_relevance_score"], results_df["pred_relevance_score"])

metrics_summary["mse_utilization"] = mean_squared_error(results_df["gold_utilization_score"],results_df["pred_utilization_score"])
metrics_summary["rmse_utilization"] = root_mean_squared_error(results_df["gold_utilization_score"],results_df["pred_utilization_score"])

metrics_summary["mse_completeness"] = mean_squared_error(results_df["gold_completeness_score"],results_df["pred_completeness_score"])
metrics_summary["rmse_completeness"] = root_mean_squared_error(results_df["gold_completeness_score"],results_df["pred_completeness_score"])

metrics_summary["accuracy_adherence"] = accuracy_score(results_df["gold_adherence_score"],results_df["pred_adherence_score"])
metrics_summary["f1_score_adherence"] = f1_score(results_df["gold_adherence_score"],results_df["pred_adherence_score"])

with open(SUMMARY_FILE, "w") as f:
    json.dump(metrics_summary, f, indent=2)

print("\n Summary saved to file.....")

shutil.copy(
      SUMMARY_FILE,
      SUMMARY_DRIVE_FILE
  )
print("\n Summary file moved to Drive.....")



 Summary saved to file.....

 Summary file moved to Drive.....
